# **Stable Diffusion** 🎨

Stable Diffusion es un modelo de difusión latente de texto a imagen creado por investigadores e ingenieros de [CompVis](https://github.com/CompVis), [Stability AI](https://stability.ai/) y [LAION](https://laion.ai/).

Está entrenado con imágenes de 512x512 de un subconjunto de la base de datos [LAION-5B](https://laion.ai/blog/laion-5b/). Este modelo utiliza un codificador de texto CLIP ViT-L/14 congelado para condicionar el modelo mediante *prompts* de texto. Con su UNet de 860M y su codificador de texto de 123M, el modelo es relativamente ligero y puede ejecutarse en muchas GPUs de consumo.

Consulta la [tarjeta del modelo](https://huggingface.co/CompVis/stable-diffusion) para más información.

Este cuaderno de Colab muestra cómo usar Stable Diffusion con la [biblioteca 🧨 Diffusers](https://github.com/huggingface/diffusers) de 🤗 Hugging Face.

¡Empecemos!

## 1. Qué es Stable Diffusion

Ahora, adentrémonos en la parte teórica de Stable Diffusion 👩‍🎓.

Stable Diffusion se basa en un tipo particular de modelo de difusión llamado **Difusión Latente** (Latent Diffusion), propuesto en [High-Resolution Image Synthesis with Latent Diffusion Models](https://arxiv.org/abs/2112.10752).

Los modelos de difusión generales son sistemas de aprendizaje automático entrenados para *eliminar el ruido* gaussiano aleatorio paso a paso, con el fin de obtener una muestra de interés, como una *imagen*. Para una visión más detallada de cómo funcionan, consulta [este colab](https://colab.research.google.com/github/huggingface/notebooks/blob/main/diffusers/diffusers_intro.ipynb).

Los modelos de difusión han demostrado lograr resultados del estado del arte en la generación de datos de imágenes. Pero una desventaja de los modelos de difusión es que el proceso inverso de eliminación de ruido es lento. Además, estos modelos consumen mucha memoria porque operan en el espacio de píxeles, lo cual se vuelve excesivamente costoso al generar imágenes de alta resolución. Por lo tanto, es un desafío entrenar estos modelos y también usarlos para inferencia.

<br>

La difusión latente puede reducir la complejidad de memoria y computación aplicando el proceso de difusión sobre un espacio _latente_ de menor dimensión, en lugar de utilizar el espacio de píxeles real. Esta es la diferencia clave entre los modelos de difusión estándar y los de difusión latente: **en la difusión latente, el modelo se entrena para generar representaciones latentes (comprimidas) de las imágenes.**

[Image of Stable Diffusion architecture diagram showing VAE, U-Net, and Text Encoder]

Hay tres componentes principales en la difusión latente:

1. Un autocodificador (VAE).
2. Una [U-Net](https://colab.research.google.com/github/huggingface/notebooks/blob/main/diffusers/diffusers_intro.ipynb#scrollTo=wW8o1Wp0zRkq).
3. Un codificador de texto, *p. ej.* el [Codificador de Texto de CLIP](

**1. El autocodificador (VAE)**

El modelo VAE consta de dos partes: un codificador y un decodificador. El codificador se utiliza para convertir la imagen en una representación latente de baja dimensión, que servirá como entrada para el modelo *U-Net*.
El decodificador, por el contrario, transforma la representación latente de nuevo en una imagen.

[Image of VAE encoder decoder architecture diagram]

Durante el _entrenamiento_ de la difusión latente, el codificador se utiliza para obtener las representaciones latentes (_latentes_) de las imágenes para el proceso de difusión hacia adelante (forward diffusion), el cual aplica más y más ruido en cada paso. Durante la _inferencia_, los latentes sin ruido generados por el proceso de difusión inverso se convierten de nuevo en imágenes utilizando el decodificador VAE.

**2. La U-Net**

La U-Net tiene una parte codificadora y una parte decodificadora, ambas compuestas por bloques ResNet.
El codificador comprime una representación de la imagen en una representación de imagen de menor resolución, y el decodificador decodifica la representación de imagen de menor resolución de vuelta a la representación de imagen original de mayor resolución, que supuestamente tiene menos ruido.
Más específicamente, la salida de la U-Net predice el residuo de ruido que se puede utilizar para calcular la representación de imagen predicha sin ruido.


Para evitar que la U-Net pierda información importante durante el submuestreo, generalmente se añaden conexiones de atajo (*short-cut connections*) entre las ResNets de submuestreo del codificador y las ResNets de sobremuestreo del decodificador.
Además, la U-Net de Stable Diffusion es capaz de condicionar su salida en embeddings de texto a través de capas de atención cruzada (*cross-attention*). Las capas de atención cruzada se añaden tanto a la parte del codificador como a la del decodificador de la U-Net, generalmente entre los bloques ResNet.

**3. El codificador de texto**

El codificador de texto es responsable de transformar el prompt de entrada, *p. ej.* "Un astronauta montando a caballo" en un espacio de embeddings que pueda ser entendido por la U-Net. Generalmente es un codificador simple *basado en transformers* que mapea una secuencia de tokens de entrada a una secuencia de embeddings de texto latentes.


Inspirado en [Imagen](https://imagen.research.google/), Stable Diffusion **no** entrena el codificador de texto durante el entrenamiento y simplemente utiliza el codificador de texto ya entrenado de CLIP, [CLIPTextModel](https://huggingface.co/docs/transformers/model_doc/clip#transformers.CLIPTextModel).

**¿Por qué es rápida y eficiente la difusión latente?**

Dado que la U-Net de los modelos de difusión latente opera en un espacio de baja dimensión, reduce enormemente los requisitos de memoria y computación en comparación con los modelos de difusión en el espacio de píxeles. Por ejemplo, el autocodificador utilizado en Stable Diffusion tiene un factor de reducción de 8. Esto significa que una imagen de forma `(3, 512, 512)` se convierte en `(3, 64, 64)` en el espacio latente, lo que requiere `8 × 8 = 64` veces menos memoria.

¡Por eso es posible generar imágenes de `512 × 512` tan rápido, incluso en GPUs de Colab de 16GB!

**Stable Diffusion durante la inferencia**

Uniendo todo, echemos ahora un vistazo más de cerca a cómo funciona el modelo en inferencia ilustrando el flujo lógico.

<p align="left">
<img src="https://raw.githubusercontent.com/patrickvonplaten/scientific_images/master/stable_diffusion.png" alt="sd-pipeline" width="500"/>
</p>

El modelo de Stable Diffusion toma como entrada tanto una semilla latente como un prompt de texto. La semilla latente se usa entonces para generar representaciones de imagen latentes aleatorias de tamaño $64 \times 64$, mientras que el prompt de texto se transforma en embeddings de texto de tamaño $77 \times 768$ a través del codificador de texto de CLIP.

A continuación, la U-Net *elimina el ruido* iterativamente de las representaciones de imagen latentes aleatorias mientras es condicionada por los embeddings de texto. La salida de la U-Net, que es el residuo de ruido, se utiliza para calcular una representación de imagen latente sin ruido mediante un algoritmo programador (*scheduler*). Se pueden utilizar muchos algoritmos de *scheduler* diferentes para este cálculo, cada uno con sus pros y sus contras. Para Stable Diffusion, recomendamos usar uno de los siguientes:

- [Scheduler PNDM](https://github.com/huggingface/diffusers/blob/main/src/diffusers/schedulers/scheduling_pndm.py) (usado por defecto).
- [Scheduler K-LMS](https://github.com/huggingface/diffusers/blob/main/src/diffusers/schedulers/scheduling_lms_discrete.py).
- [Scheduler Heun Discrete](https://github.com/huggingface/diffusers/blob/main/src/diffusers/schedulers/scheduling_heun_discrete.py).
- [Scheduler DPM Solver Multistep](https://github.com/huggingface/diffusers/blob/main/src/diffusers/schedulers/scheduling_dpmsolver_multistep.py). Este *scheduler* es capaz de lograr una gran calidad en menos pasos. ¡Puedes probar con 25 en lugar de los 50 predeterminados!

La teoría sobre cómo funciona el algoritmo del *scheduler* está fuera del alcance de este cuaderno, pero en resumen se debe recordar que calculan la representación de imagen sin ruido predicha a partir de la representación de ruido anterior y el residuo de ruido predicho.
Para más información, recomendamos consultar [Elucidating the Design Space of Diffusion-Based Generative Models](https://arxiv.org/abs/2206.00364)

El proceso de *eliminación de ruido* se repite *aprox.* 50 veces para recuperar paso a paso mejores representaciones de imagen latentes.
Una vez completado, la representación de imagen latente es decodificada por la parte decodificadora del auto codificador variacional (VAE).

## 2. Cómo usar `StableDiffusionPipeline`

Ahora que conocemos los aspectos teóricos de cómo funciona Stable Diffusion,
¡vamos a probarlo un poco 🤗!

En esta sección, mostramos cómo puedes ejecutar inferencia de texto a imagen ¡en solo unas pocas líneas de código!

### Configuración

Primero, asegúrate de estar usando un entorno de ejecución de GPU para ejecutar este cuaderno, para que la inferencia sea mucho más rápida. Si el siguiente comando falla, usa el menú `Entorno de ejecución` (Runtime) de arriba y selecciona `Cambiar tipo de entorno de ejecución`.

In [ ]:
import torch

In [ ]:
if torch.cuda.is_available():
    # Shows the nVidia GPUs, if this system has any
    !nvidia-smi

A continuación, debes instalar `diffusers` así como `scipy`, `ftfy` y `transformers`. Se utiliza `accelerate` para lograr una carga mucho más rápida.

In [ ]:
!pip install diffusers==0.11.1
!pip install transformers scipy ftfy accelerate

### Pipeline de Stable Diffusion

`StableDiffusionPipeline` es un pipeline de inferencia de extremo a extremo que puedes usar para generar imágenes a partir de texto con solo unas pocas líneas de código.

Primero, cargamos los pesos pre-entrenados de todos los componentes del modelo. En este cuaderno usamos la versión 1.4 de Stable Diffusion ([CompVis/stable-diffusion-v1-4](https://huggingface.co/CompVis/stable-diffusion-v1-4)), pero hay otras variantes que quizás quieras probar:
* [runwayml/stable-diffusion-v1-5](https://huggingface.co/runwayml/stable-diffusion-v1-5)
* [stabilityai/stable-diffusion-2-1-base](https://huggingface.co/stabilityai/stable-diffusion-2-1-base)
* [stabilityai/stable-diffusion-2-1](https://huggingface.co/stabilityai/stable-diffusion-2-1). Esta versión puede producir imágenes con una resolución de 768x768, mientras que las otras funcionan a 512x512.

Además del id del modelo [CompVis/stable-diffusion-v1-4](https://huggingface.co/CompVis/stable-diffusion-v1-4), también estamos pasando una `revision` y `torch_dtype` específicos al método `from_pretrained`.

Queremos asegurarnos de que cualquier Google Colab gratuito pueda ejecutar Stable Diffusion, por lo tanto, cargamos los pesos desde la rama de media precisión [`fp16`](https://huggingface.co/CompVis/stable-diffusion-v1-4/tree/fp16) y también le indicamos a `diffusers` que espere los pesos en precisión float16 pasando `torch_dtype=torch.float16`.

Si deseas asegurar la mayor precisión posible, asegúrate de eliminar `torch_dtype=torch.float16` a costa de un mayor uso de memoria.

In [ ]:
# This is added to get around some issues of Torch not loading models correctly (test on Mac OS X and Kubuntu Linux)
!pip install --upgrade huggingface-hub==0.26.2 transformers==4.46.1 tokenizers==0.20.1 diffusers==0.31.0

In [ ]:
!pip install peft==0.17.1 transformers==4.49.0 accelerate diffusers --upgrade

In [ ]:
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4", torch_dtype=torch.float16)

A continuación, movamos el pipeline a la GPU para tener una inferencia más rápida.

In [ ]:
if torch.cuda.is_available():
    device=torch.device("cuda")
elif torch.backends.mps.is_available():
    device=torch.device("mps")

pipe = pipe.to(device)

Y ya tenemos todo listo para empezar a generar imágenes:

In [ ]:
prompt = "a photograph of an astronaut riding a horse"
image = pipe(prompt).images[0]  # image here is in [PIL format](https://pillow.readthedocs.io/en/stable/)

# Now to display an image you can either save it such as:
image.save(f"astronaut_rides_horse.png")

# or if you're in a google colab you can directly display it with
image

Ejecutar la celda anterior varias veces te dará una imagen diferente cada vez. Si quieres una salida determinista, puedes pasar una semilla aleatoria al pipeline. Cada vez que uses la misma semilla, obtendrás el mismo resultado de imagen.

In [ ]:
generator = torch.Generator(device).manual_seed(1024)

image = pipe(prompt, generator=generator).images[0]

image

Puedes cambiar el número de pasos de inferencia utilizando el argumento `num_inference_steps`. En general, los resultados son mejores cuantos más pasos utilices. Stable Diffusion, al ser uno de los modelos más recientes, funciona de maravilla con un número relativamente pequeño de pasos, por lo que recomendamos usar el valor por defecto de `50`. Si deseas resultados más rápidos, puedes usar un número menor.

La siguiente celda utiliza la misma semilla que antes, pero con menos pasos. Observa cómo algunos detalles, como la cabeza del caballo o el casco, son menos realistas y están menos definidos que en la imagen anterior:

In [ ]:
generator = torch.Generator(device).manual_seed(1024)

image = pipe(prompt, num_inference_steps=15, generator=generator).images[0]

image

El otro parámetro en la llamada al pipeline es `guidance_scale`. Es una forma de aumentar la adherencia a la señal condicional (que en este caso es el texto), así como la calidad general de la muestra. En términos simples, la guía sin clasificador (*classifier-free guidance*) obliga a que la generación coincida mejor con el prompt. Valores como `7` u `8.5` ofrecen buenos resultados; si usas un número muy elevado, las imágenes pueden verse bien, pero serán menos diversas.

Puedes aprender sobre los detalles técnicos de este parámetro en [la última sección](https://colab.research.google.com/drive/1ALXuCM5iNnJDNW5vqBm5lCtUQtZJHN2f?authuser=1#scrollTo=UZp-ynZLrS-S) de este cuaderno.

Para generar varias imágenes con el mismo prompt, simplemente utilizamos una lista con el mismo prompt repetido varias veces. Enviaremos la lista al pipeline en lugar de la cadena de texto (string) que usamos anteriormente.

Primero, vamos a escribir una función auxiliar para mostrar una cuadrícula de imágenes. Simplemente ejecuta la siguiente celda para crear la función `image_grid`, o despliega el código si te interesa saber cómo está hecha.

In [ ]:
from PIL import Image

def image_grid(imgs, rows, cols):
    assert len(imgs) == rows*cols

    w, h = imgs[0].size
    grid = Image.new('RGB', size=(cols*w, rows*h))
    grid_w, grid_h = grid.size

    for i, img in enumerate(imgs):
        grid.paste(img, box=(i%cols*w, i//cols*h))
    return grid

Ahora, podemos generar una imagen en cuadrícula una vez que hayamos ejecutado el pipeline con una lista de 3 prompts.

In [ ]:
num_images = 3
prompt = ["a photograph of an astronaut riding a horse"] * num_images

images = pipe(prompt).images

grid = image_grid(images, rows=1, cols=3)
grid

Y aquí se muestra cómo generar una cuadrícula de `n × m` imágenes.

In [ ]:
num_cols = 3
num_rows = 4

prompt = ["a photograph of an astronaut riding a horse"] * num_cols

all_images = []
for i in range(num_rows):
  images = pipe(prompt).images
  all_images.extend(images)

grid = image_grid(all_images, rows=num_rows, cols=num_cols)
grid

### Generar imágenes que no sean cuadradas

Stable Diffusion produce imágenes de `512 × 512` píxeles por defecto. Pero es muy fácil anular este valor predeterminado usando los argumentos `height` (altura) y `width` (anchura), de modo que puedes crear imágenes rectangulares en formato vertical u horizontal.

Estas son algunas recomendaciones para elegir buenos tamaños de imagen:
- Asegúrate de que tanto `height` como `width` sean múltiplos de `8`.
- Bajar de 512 puede dar lugar a imágenes de menor calidad.
- Superar los 512 en ambas direcciones repetirá áreas de la imagen (se pierde la coherencia global).
- La mejor manera de crear imágenes que no sean cuadradas es usar `512` en una dimensión y un valor mayor en la otra.

In [ ]:
prompt = "a photograph of an astronaut riding a horse"

image = pipe(prompt, height=512, width=768).images[0]
image